# Reddit Data Collection with Selenium

Searches comments across Reddit, without a subreddit restriction, for each lexicon word and keeps up to 3 distinct Persian comments per word, read from their full comment pages. No API key, archive service or automatic login is used.

Reddit can require login, show a CAPTCHA, rate limit or deny access, so three matches per word are not guaranteed. The browser pauses so a CAPTCHA can be solved manually; a block that remains stops the collection and saves progress. Search covers Reddit's search index, not every public comment.

The input is a JSON list of words or `{"word": [...]}`. `RedditComments.csv` holds the cleaned and original text with the word, comment ID and URL, subreddit, author, time and score. `RedditComments.progress.json` is a checkpoint saved after each accepted comment and is used to resume. `RedditWordStatus.csv` records the count and status of each word; `no_match` means no qualifying match within the configured limits. Persian detection is a script and word heuristic, so short text shared by Persian and Arabic can remain ambiguous.

## 1. Dependencies

Chrome or Edge must be installed. Selenium Manager downloads a matching driver on first use.

In [ ]:
%pip install "selenium>=4.26,<5" "pandas>=2.2,<4"

In [ ]:
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urlparse

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

## 2. Configuration

Set `BROWSER` to `edge` or `chrome`, and `DRIVER_PATH = None` to let Selenium Manager download a matching driver. Use a small `RUN_LIMIT` for a test run and `None` for the full lexicon. The browser uses its own profile in `BROWSER_PROFILE`, separate from your usual one. Do not run two collectors with the same profile or checkpoint at the same time.

In [ ]:
WORDS_FILE = Path("Only_Persian_Offensive_lexicon.json")
OUTPUT_CSV = Path("RedditComments.csv")
PROGRESS_FILE = Path("RedditComments.progress.json")
STATUS_CSV = Path("RedditWordStatus.csv")
DIAGNOSTICS_DIR = Path(".reddit_diagnostics")
COMMENTS_PER_WORD = 3
EXCLUDE_BOTS = False                # True drops AutoModerator and *bot authors
RUN_LIMIT = None
BROWSER = "edge"                    # "edge" or "chrome"
DRIVER_PATH = None
# DRIVER_PATH = None lets Selenium Manager download a matching driver.
BROWSER_PROFILE = Path(".reddit_browser_profile")
SELENIUM_CACHE = Path(".selenium_cache")
MANUAL_CHALLENGES = True             # False: save and stop immediately on a challenge
CHALLENGE_TIMEOUT = 300             # seconds to solve a CAPTCHA manually
CHALLENGE_POLL_INTERVAL = 2         # seconds
CHALLENGE_STATUS_INTERVAL = 15      # seconds
SEARCH_SORTS = ("relevance", "new")
MAX_SEARCH_SCROLLS = 4
MAX_CANDIDATES_PER_QUERY = 15
PAGE_TIMEOUT = 30
SCROLL_WAIT = 3
DELAY_BETWEEN_PAGES = 6
RATE_LIMIT_WAIT = 60                # seconds; multiplied by the retry number after HTTP 429
RATE_LIMIT_RETRIES = 3
DELAY_BETWEEN_WORDS = 6
MAX_CONSECUTIVE_ERRORS = 3
MAX_BROWSER_RESTARTS = 2            # per word
RESTART_WAIT = 60                   # seconds
CSV_COLUMNS = ["word", "comment_id", "subreddit", "author", "created_at", "score",
               "comment_url", "raw_text", "text"]
STATUS_COLUMNS = ["word", "status", "count", "message", "updated_at"]
_last_navigation_at = 0.0

## 3. Read words and filter Persian text

Matching uses whole words, so `کس` does not match `عکس`. Arabic and Persian forms of ی and ک, diacritics and half spaces are normalized before comparison, and a half space counts as a word boundary. Matching runs on the cleaned text, so a removed username cannot be the only match. The raw text is kept separately.

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(
        str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ـ": "", "\u200c": " "}))
    text = "".join(c for c in text if unicodedata.category(c) not in {"Mn", "Me", "Cf"})
    return " ".join(text.casefold().split())


def load_words(path):
    with Path(path).open(encoding="utf-8-sig") as handle:
        data = json.load(handle)
    values = data.get("word") if isinstance(data, dict) else data
    if not isinstance(values, list) or not values:
        raise ValueError('Expected a non-empty JSON list or {"word": ["..."]}.')
    words, seen = [], set()
    for index, word in enumerate(values):
        if not isinstance(word, str) or not normalize_text(word):
            raise ValueError(f"Word at index {index} must be a non-empty string.")
        key = normalize_text(word)
        if not any(c.isalpha() for c in key):
            raise ValueError(f"Word at index {index} contains no letters.")
        if key not in seen:
            seen.add(key)
            words.append(word.strip())
    return words


def contains_word(text, word):
    parts = normalize_text(word).split()
    if not parts:
        return False
    pattern = r"(?<!\w)" + r"\s*".join(re.escape(part) for part in parts) + r"(?!\w)"
    return re.search(pattern, normalize_text(text)) is not None


PERSIAN_HINTS = {"شما", "برای", "این", "اون", "آنها", "هست", "هستم", "هستی", "نیست",
                 "نیستم", "خیلی", "چرا", "چون", "میشه", "نمیشه", "آره", "باشه", "بشه",
                 "دارم", "داری", "داره", "کردم", "کردی", "کرده", "خودم", "همین", "همون",
                 "که", "از", "را", "است", "یه", "بود", "می", "اینا", "اونا", "چی"}
# Colloquial Arabic (e.g. Iraqi) also uses گ/چ/پ, so these words count against Persian.
ARABIC_HINTS = {normalize_text(word) for word in (
    "انا", "انت", "هذا", "هذه", "لان", "راح", "شنو", "شلون", "اكو", "ماكو", "ليش",
    "هيك", "مش", "عشان", "اللي", "ايش", "وش", "اذا", "كان", "كانت", "ولا")}
OTHER_SCRIPT_LETTERS = set("ٹڈڑںھہۓےڕڵۆێەټځڅډړږښڼۍې")


def is_persian(text):
    # Script/word heuristic, not a language model.
    original = unicodedata.normalize("NFKC", text)
    if any(c in OTHER_SCRIPT_LETTERS for c in original):
        return False
    normalized = normalize_text(original)
    letters = [c for c in normalized if unicodedata.category(c).startswith("L")]
    script = [c for c in letters if "\u0600" <= c <= "\u06ff"]
    if len(script) < 3 or len(script) / max(len(letters), 1) < 0.60:
        return False
    persian_markers = sum(c in "پچژگکی" for c in original)
    arabic_markers = sum(c in "ةى" for c in original)
    tokens = set(re.findall(r"[^\W\d_]+", normalized))
    persian_hits = len(tokens & PERSIAN_HINTS)
    arabic_hits = len(tokens & ARABIC_HINTS)
    return (arabic_markers == 0 and arabic_hits <= persian_hits
            and (persian_markers > 0 or persian_hits > 0))


def clean_text(text):
    text = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", text)
    text = re.sub(r"(?<!\w)(?:@|/?u/)[^\s]+", "", text)
    text = text.replace("#", "").replace("*", "").replace(">", "").replace(",", " ")
    return " ".join(text.split())


def is_bot(author):
    return author.casefold() == "automoderator" or author.casefold().endswith("bot")


def comment_row(word, node):
    raw = node.get("body", "").strip()
    text = clean_text(raw)
    if (normalize_text(raw) in {"[deleted]", "[removed]"} or not text
            or (EXCLUDE_BOTS and is_bot(node.get("author", ""))) or not contains_word(text, word)
            or not is_persian(text)):
        return None
    return {"word": word, "comment_id": node["comment_id"], "subreddit": node["subreddit"],
            "author": node.get("author", ""), "created_at": node.get("created_at", ""),
            "score": node.get("score", ""), "comment_url": node["comment_url"],
            "raw_text": raw, "text": text}

In [ ]:
words = load_words(WORDS_FILE)
selected_words = words if RUN_LIMIT is None else words[:RUN_LIMIT]
print(f"Loaded {len(words)} unique words; selected {len(selected_words)}.")
print("Output:", OUTPUT_CSV.resolve())

## 4. Browser and access checks

If Reddit shows a CAPTCHA, the run prints `[WAITING]` and checks the page every few seconds until it is solved in the browser window, for up to `CHALLENGE_TIMEOUT` seconds. After that, progress is saved and the run stops.

Log messages: `[SEARCH]` searching for links, `[READ]` reading a full comment, `[FILTERED]` comment rejected, `[SAVED]` comment written to the checkpoint and CSV, `[RATE LIMIT]` Reddit returned HTTP 429 and the page is reloaded after a wait, `[RESTART]` the browser stopped responding and was restarted, `[SKIPPED]` a link that froze the browser is left for the Retry cell.

In [ ]:
class RedditAccessBlocked(RuntimeError):
    """The visible browser needs human action or Reddit denies access."""


class RedditPageLoadError(TimeoutException):
    """Reddit did not render comment content; stop instead of losing more candidates."""


class BrowserUnresponsive(RedditPageLoadError):
    """The browser tab stopped responding; only a browser restart recovers it."""


def start_browser():
    if BROWSER not in {"chrome", "edge"}:
        raise ValueError('BROWSER must be "chrome" or "edge".')
    SELENIUM_CACHE.mkdir(parents=True, exist_ok=True)
    os.environ["SE_CACHE_PATH"] = str(SELENIUM_CACHE.resolve())
    options = webdriver.ChromeOptions() if BROWSER == "chrome" else webdriver.EdgeOptions()
    options.add_argument("--user-data-dir=" + str(BROWSER_PROFILE.resolve()))
    options.add_argument("--no-first-run")
    options.add_argument("--lang=en-US")
    options.add_experimental_option("prefs", {"intl.accept_languages": "en-US,en"})
    options.page_load_strategy = "eager"
    kwargs = {"options": options}
    if DRIVER_PATH:
        if BROWSER == "chrome":
            from selenium.webdriver.chrome.service import Service
        else:
            from selenium.webdriver.edge.service import Service
        kwargs["service"] = Service(str(Path(DRIVER_PATH).resolve()))
    try:
        browser = webdriver.Chrome(**kwargs) if BROWSER == "chrome" else webdriver.Edge(**kwargs)
    except WebDriverException as error:
        raise RuntimeError(
            f"Could not start {BROWSER}. Close another collector using {BROWSER_PROFILE}; "
            "check driver-download internet access or set DRIVER_PATH to a matching driver. "
            f"Original error: {error.msg}") from error
    browser.set_page_load_timeout(PAGE_TIMEOUT)
    return browser


def restart_browser(driver):
    try:
        driver.quit()
    except WebDriverException as error:
        print("Previous browser did not close cleanly:", error.msg[:150], flush=True)
    return start_browser()


def page_text(driver):
    return driver.find_element(By.TAG_NAME, "body").text


def access_issue(driver):
    body = page_text(driver).casefold()
    title = driver.title.casefold()
    path = urlparse(driver.current_url).path
    if path.startswith(("/login", "/account/login")):
        return "Reddit is requesting login; you may log in manually or stop."
    if any(s in title for s in ("just a moment", "security check", "verify you are")):
        return "Browser verification/CAPTCHA is required."
    frames = driver.find_elements(By.CSS_SELECTOR,
        'iframe[src*="recaptcha"], iframe[src*="hcaptcha"], iframe[src*="challenges.cloudflare"]')
    if any(frame.is_displayed() for frame in frames):
        return "A visible CAPTCHA is present."
    # Ordinary Log In buttons and quoted comment text are not full-page access blocks.
    if len(body) < 6000:
        for marker in ("blocked by network security", "your request has been blocked",
                       "whoa there, pardner", "too many requests", "you are doing that too much",
                       "verify you are human", "confirm you're human", "you've been blocked",
                       "prove your humanity", "complete the challenge below"):
            if marker in body:
                content = driver.find_elements(By.CSS_SELECTOR, "shreddit-comment, .thing.comment")
                if not content and not comment_links(driver):
                    return f"Reddit access check: {marker}."
    return None


def check_access(driver):
    issue = access_issue(driver)
    if not issue:
        return
    if not MANUAL_CHALLENGES:
        raise RedditAccessBlocked(issue)
    print("[WAITING]", issue, "\nURL:", driver.current_url, flush=True)
    print("Solve the challenge in the SAME browser window. No Enter is needed. "
          "Collection resumes automatically when the access check clears. "
          "Use Interrupt/Stop to stop safely.", flush=True)
    deadline = time.monotonic() + CHALLENGE_TIMEOUT
    next_status = time.monotonic() + CHALLENGE_STATUS_INTERVAL
    while True:
        remaining = access_issue(driver)
        if not remaining:
            print("[RESUMED] Access check cleared; continuing collection.", flush=True)
            return
        now = time.monotonic()
        if now >= deadline:
            raise RedditAccessBlocked(
                f"Manual access check was not cleared within {CHALLENGE_TIMEOUT:g} seconds. "
                "Progress will be saved; resolve access in the browser and rerun collection.")
        if now >= next_status:
            print(f"[WAITING] Still waiting for manual verification; "
                  f"{max(0, int(deadline - now))} seconds left. No extraction during this wait.", flush=True)
            next_status = now + CHALLENGE_STATUS_INTERVAL
        time.sleep(min(CHALLENGE_POLL_INTERVAL, deadline - now))



def is_error_page(driver):
    # The browser replaces a failed response (usually HTTP 429) with its own error page.
    if driver.current_url.startswith("chrome-error://"):
        return True
    body = page_text(driver).casefold()
    return len(body) < 6000 and any(marker in body for marker in (
        "http error 429", "too many requests", "whoa there, pardner"))


def navigate(driver, url):
    global _last_navigation_at
    for attempt in range(RATE_LIMIT_RETRIES + 1):
        delay = DELAY_BETWEEN_PAGES - (time.monotonic() - _last_navigation_at)
        if delay > 0:
            time.sleep(delay)
        _last_navigation_at = time.monotonic()
        try:
            driver.get(url)
        except TimeoutException as error:
            # "Timed out receiving message from renderer": a new tab hangs too, so restart.
            raise BrowserUnresponsive(
                f"Page did not load within {PAGE_TIMEOUT}s; the browser tab stopped responding.") from error
        WebDriverWait(driver, PAGE_TIMEOUT).until(
            lambda d: d.execute_script("return document.readyState") in {"interactive", "complete"})
        if not is_error_page(driver):
            check_access(driver)
            return
        if attempt == RATE_LIMIT_RETRIES:
            break
        wait = RATE_LIMIT_WAIT * (attempt + 1)
        print(f"[RATE LIMIT] Reddit returned an error page (usually HTTP 429). Waiting {wait}s, "
              f"then retry {attempt + 1}/{RATE_LIMIT_RETRIES}.", flush=True)
        time.sleep(wait)
    raise RedditAccessBlocked(
        f"Reddit kept returning an error page (usually HTTP 429) for {url}. "
        "Progress is saved; wait several minutes and rerun collection.")


def parse_comment_url(url):
    if url.startswith("/r/"):
        url = "https://www.reddit.com" + url
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {
            "reddit.com", "www.reddit.com", "old.reddit.com", "new.reddit.com", "m.reddit.com"}:
        return None
    match = re.fullmatch(r"/r/([^/]+)/comments/([a-z0-9]+)/([^/]+)/([a-z0-9]+)/?", parsed.path, re.I)
    if not match:
        return None
    subreddit, post_id, slug, comment_id = match.groups()
    return {"subreddit": subreddit, "comment_id": comment_id.lower(),
            "comment_url": f"https://www.reddit.com/r/{subreddit}/comments/{post_id}/comment/{comment_id}/"}


def search_variants(word):
    variants = [word.strip(), normalize_text(word)]
    if "\u200c" in word:
        variants.append(normalize_text(word).replace(" ", ""))
    return list(dict.fromkeys(variants))


def comment_links(driver):
    # Read links in the document and any open Web Component shadow roots.
    urls = driver.execute_script("""
        const roots = [document], urls = [];
        for (let i = 0; i < roots.length; i++) {
            for (const el of roots[i].querySelectorAll('*')) {
                if (el.shadowRoot) roots.push(el.shadowRoot);
                if (el.tagName === 'A' && el.href && el.href.includes('/comments/')) urls.push(el.href);
            }
        }
        return urls;
    """)
    found = {}
    for url in urls:
        parsed = parse_comment_url(url)
        if parsed:
            found.setdefault(parsed["comment_id"], parsed["comment_url"])
    return list(found.values())


def search_is_empty(driver):
    text = page_text(driver).casefold()
    return any(s in text for s in ("no results", "no search results", "couldn't find any results",
                                   "couldn’t find any results", "there doesn't seem to be anything here"))


def find_comment_links(driver, word, sort):
    query = '"' + word.replace('"', ' ') + '"'
    # Search all of Reddit, not one subreddit.
    url = "https://www.reddit.com/search/?" + urlencode({"q": query, "type": "comments", "sort": sort, "t": "all"})
    print(f"[SEARCH] Query: {word!r}; sort: {sort}", flush=True)
    navigate(driver, url)
    print("[SEARCH] Page opened; waiting for comment results.", flush=True)

    def ready(browser):
        check_access(browser)
        return comment_links(browser) or search_is_empty(browser)

    WebDriverWait(driver, PAGE_TIMEOUT).until(ready,
        "No comment results or explicit empty state. Reddit may have changed its page layout.")
    links = dict.fromkeys(comment_links(driver))
    print(f"[SEARCH] Found {len(links)} comment links.", flush=True)
    if not links and search_is_empty(driver):
        return []
    for scroll_index in range(MAX_SEARCH_SCROLLS):
        if len(links) >= MAX_CANDIDATES_PER_QUERY:
            break
        print(f"[SEARCH] Scroll {scroll_index + 1}/{MAX_SEARCH_SCROLLS}; "
              f"{len(links)} links found.", flush=True)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight)")
        previous = len(links)
        try:
            WebDriverWait(driver, SCROLL_WAIT).until(
                lambda d: access_issue(d) or any(url not in links for url in comment_links(d)))
        except TimeoutException:
            break  # A bounded scroll without new links is a normal end of available results.
        check_access(driver)
        links.update(dict.fromkeys(comment_links(driver)))
        if len(links) == previous:
            break
    candidates = list(links)[:MAX_CANDIDATES_PER_QUERY]
    print(f"[SEARCH] Inspecting up to {len(candidates)} full comments.", flush=True)
    return candidates

## 5. Read full comments

Search results are used only to find comment links; each comment is read from its own page by its exact ID, never from a snippet, post or parent comment. Legacy links are converted to `/r/<subreddit>/comments/<post_id>/comment/<comment_id>/` before opening. Links that cannot be read stay in the checkpoint under `pending_comments` and are retried before the next search for that word. Bot comments are kept unless `EXCLUDE_BOTS = True`.

In [ ]:
COMMENT_SCRIPT = r"""
const target = arguments[0], roots = [document];
for (let i = 0; i < roots.length; i++) {
    for (const el of roots[i].querySelectorAll('*')) if (el.shadowRoot) roots.push(el.shadowRoot);
}
const all = selector => roots.flatMap(root => Array.from(root.querySelectorAll(selector)));
const plainText = body => {
    // textContent preserves unclipped text; explicit separators preserve paragraph boundaries.
    const copy = body.cloneNode(true);
    for (const br of copy.querySelectorAll('br')) br.replaceWith('\n');
    for (const block of copy.querySelectorAll('p, div, li, blockquote, pre, h1, h2, h3')) block.append('\n');
    return copy.textContent.trim();
};
for (const node of all('shreddit-comment')) {
    const id = (node.getAttribute('thingid') || node.id || '').replace(/^(thing_)?t1_/, '');
    if (id !== target) continue;
    const bodies = Array.from(node.querySelectorAll('[slot="comment"], [id$="-comment-rtjson-content"], .md'));
    const body = bodies.find(el => el.closest('shreddit-comment') === node);
    if (!body) return null;
    const timestamp = Array.from(node.querySelectorAll('time')).find(el => el.closest('shreddit-comment') === node);
    return {body: plainText(body), author: node.getAttribute('author') || '',
            created_at: node.getAttribute('created') || (timestamp ? timestamp.getAttribute('datetime') : '') || '',
            score: node.getAttribute('score') || ''};
}
// Legacy layout support, if Reddit serves it for the same permalink.
for (const node of all('.thing.comment')) {
    const id = (node.getAttribute('data-fullname') || node.id || '').replace(/^(thing_)?t1_/, '');
    if (id !== target) continue;
    const body = Array.from(node.querySelectorAll('.usertext-body .md')).find(
        el => el.closest('.thing.comment') === node);
    if (!body) return null;
    const owned = selector => Array.from(node.querySelectorAll(selector)).find(el => el.closest('.thing.comment') === node);
    const author = owned('.author'), timestamp = owned('time');
    const score = owned('.score.unvoted, .score');
    return {body: plainText(body), author: author ? author.textContent.trim() : '',
            created_at: timestamp ? timestamp.getAttribute('datetime') || '' : '',
            score: score ? score.getAttribute('title') || score.textContent.trim() : ''};
}
return null;
"""


def mark_pending_comment(state, word, url, error="", attempt=False, hung=False):
    identity = parse_comment_url(url)
    if identity is None:
        raise ValueError(f"Not a comment permalink: {url}")
    url = identity["comment_url"]
    pending = state.setdefault("pending_comments", {}).setdefault(normalize_text(word), {})
    entry = pending.setdefault(url, {"word": word, "comment_url": url, "attempts": 0,
                                     "last_error": "", "updated_at": utc_now()})
    if attempt:
        entry["attempts"] += 1
    if error:
        entry["last_error"] = str(error)[:2000]
    if hung:
        entry["browser_hang"] = True
    entry["updated_at"] = utc_now()


def resolve_pending_comment(state, word, url):
    identity = parse_comment_url(url)
    if identity is None:
        return
    queue = state.get("pending_comments", {})
    key = normalize_text(word)
    pending = queue.get(key, {})
    pending.pop(identity["comment_url"], None)
    if not pending:
        queue.pop(key, None)


COMMENT_DIAGNOSTIC_SCRIPT = r"""
const target = arguments[0], roots = [document];
for (let i = 0; i < roots.length; i++) {
    for (const el of roots[i].querySelectorAll('*')) if (el.shadowRoot) roots.push(el.shadowRoot);
}
const all = selector => roots.flatMap(root => Array.from(root.querySelectorAll(selector)));
const comments = all('shreddit-comment,.thing.comment');
return {url: location.href, title: document.title, ready_state: document.readyState,
        target_id: target, comment_count: comments.length,
        comment_ids: comments.slice(0, 30).map(el => el.getAttribute('thingid') || el.getAttribute('data-fullname') || el.id),
        post_count: all('shreddit-post,.thing.link').length,
        main_count: all('main').length, body_preview: document.body.innerText.slice(0, 1500)};
"""


def save_comment_diagnostic(driver, identity):
    diagnostic = driver.execute_script(COMMENT_DIAGNOSTIC_SCRIPT, identity["comment_id"])
    diagnostic["kind"] = ("empty_shell" if diagnostic["comment_count"] == 0
                          and diagnostic["post_count"] == 0 and diagnostic["main_count"] == 0
                          else "target_not_found")
    diagnostic["captured_at"] = utc_now()
    folder = Path(DIAGNOSTICS_DIR)
    folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    path = folder / f"{identity['comment_id']}_{stamp}.json"
    path.write_text(json.dumps(diagnostic, ensure_ascii=False, indent=2), encoding="utf-8")
    return diagnostic, path


def read_full_comment(driver, url):
    identity = parse_comment_url(url)
    if not identity:
        raise ValueError(f"Not a Reddit comment permalink: {url}")
    # Legacy /post-slug/comment-id/ routes can render only an empty shell.
    navigate(driver, identity["comment_url"] + "?context=3")

    def ready(browser):
        check_access(browser)
        return browser.execute_script(COMMENT_SCRIPT, identity["comment_id"])

    try:
        node = WebDriverWait(driver, PAGE_TIMEOUT).until(ready,
            f"Full comment {identity['comment_id']} not found.")
    except TimeoutException as error:
        diagnostic, path = save_comment_diagnostic(driver, identity)
        message = (f"Comment {identity['comment_id']}: {diagnostic['kind']}; "
                   f"{diagnostic['comment_count']} comment nodes loaded. Diagnostic: {path}. "
                   "This does not prove the comment was deleted.")
        if diagnostic["kind"] == "empty_shell":
            raise RedditPageLoadError(message) from error
        raise TimeoutException(message) from error
    return {**node, **identity}


def iter_word_comments(driver, word, state=None):
    pending = state.get("pending_comments", {}).get(normalize_text(word), {}) if state is not None else {}
    hung_urls = [url for url, entry in pending.items() if entry.get("browser_hang")]
    # Links that froze the browser stay in the checkpoint; only the Retry cell reopens them.
    seen_ids = {parse_comment_url(url)["comment_id"] for url in hung_urls}
    if hung_urls:
        print(f"[SKIPPED] {len(hung_urls)} links that froze the browser are not reopened.", flush=True)
    failed_urls = []
    consecutive_unreadable = 0

    def candidate_urls():
        retry_urls = [url for url in pending if url not in hung_urls]
        if retry_urls:
            print(f"[RETRY] {len(retry_urls)} previously unreadable comment links queued for this word.", flush=True)
            yield from retry_urls
        for variant in search_variants(word):
            for sort in SEARCH_SORTS:
                yield from find_comment_links(driver, variant, sort)

    for candidate_index, url in enumerate(candidate_urls(), 1):
        identity = parse_comment_url(url)
        comment_id = identity["comment_id"]
        if comment_id in seen_ids:
            continue
        seen_ids.add(comment_id)
        url = identity["comment_url"]
        if state is not None:
            mark_pending_comment(state, word, url, attempt=True)
            save_progress(state)
        print(f"[READ] Candidate {candidate_index}: {url}", flush=True)
        try:
            node = read_full_comment(driver, url)
        except TimeoutException as error:
            failed_urls.append(url)
            consecutive_unreadable += 1
            if state is not None:
                mark_pending_comment(state, word, url, error=error,
                                     hung=isinstance(error, BrowserUnresponsive))
                save_progress(state)
            print(f"[DEFERRED] {url} | {error.msg[:400]}", flush=True)
            if isinstance(error, RedditPageLoadError):
                raise
            if consecutive_unreadable >= MAX_CONSECUTIVE_ERRORS:
                raise RedditPageLoadError("Repeated unreadable comment pages; stopping the run. "
                                          "Failed URLs remain queued for retry. " + str(error)) from error
            continue
        consecutive_unreadable = 0
        print(f"[READ] Extracted {len(node.get('body', ''))} characters.", flush=True)
        yield node
    if failed_urls:
        raise TimeoutException(f"{len(failed_urls)} comment pages could not be read; "
                               "failed URLs remain queued for retry. Example: " + failed_urls[0])


## 6. Checkpoint and collect

The JSON checkpoint is the source of truth and the CSV files are rebuilt from it. Rerunning the collection cell continues with words that have not been searched yet. Words left incomplete (`partial`, `no_match`, `error`) are retried only by the Retry cell. A comment counts once per word but can match several words. Finding a lexicon word does not label a comment as offensive.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def load_progress():
    if not Path(PROGRESS_FILE).exists():
        if Path(OUTPUT_CSV).exists():
            raise ValueError(f"{OUTPUT_CSV} exists without {PROGRESS_FILE}. Choose new output paths "
                             "or back up the existing CSV before starting a fresh collection.")
        return {"version": 1, "comments": [], "words": {}}
    with Path(PROGRESS_FILE).open(encoding="utf-8") as handle:
        state = json.load(handle)
    if (not isinstance(state, dict) or state.get("version") != 1
            or not isinstance(state.get("comments"), list) or not isinstance(state.get("words"), dict)
            or not isinstance(state.get("pending_comments", {}), dict)):
        raise ValueError("Invalid or incompatible checkpoint; keep a backup and use a new output path.")
    for row in state["comments"]:
        if not isinstance(row, dict) or any(column not in row for column in CSV_COLUMNS):
            raise ValueError("Incomplete comment row in checkpoint; do not overwrite it.")
    return state


def save_progress(state):
    # Atomic replacement. If a CSV write fails, accepted rows remain in the authoritative JSON.
    progress = Path(PROGRESS_FILE)
    progress.parent.mkdir(parents=True, exist_ok=True)
    temporary = progress.with_name(progress.name + ".tmp")
    temporary.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(progress)
    exports = [(Path(OUTPUT_CSV), pd.DataFrame(state["comments"], columns=CSV_COLUMNS)),
               (Path(STATUS_CSV), pd.DataFrame(list(state["words"].values()), columns=STATUS_COLUMNS))]
    for path, frame in exports:
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        frame.to_csv(temporary, index=False, encoding="utf-8-sig")
        temporary.replace(path)


def rows_for_word(state, word):
    key = normalize_text(word)
    return [row for row in state["comments"] if normalize_text(row["word"]) == key]


def record_status(state, word, status, message=""):
    state["words"][normalize_text(word)] = {
        "word": word, "status": status, "count": len(rows_for_word(state, word)),
        "message": message, "updated_at": utc_now()}


def unsearched_words(state, words):
    """Words not yet finished; incomplete words (partial, no_match, error) are left for the Retry cell."""
    return [word for word in words
            if state["words"].get(normalize_text(word), {}).get("status") in {None, "interrupted", "blocked"}]


def collect_word(driver, word, state):
    existing = rows_for_word(state, word)
    checked = 0
    ids = {row["comment_id"] for row in existing}
    texts = {normalize_text(row["text"]) for row in existing}
    for node in iter_word_comments(driver, word, state):
        checked += 1
        row = comment_row(word, node)
        if row is None:
            resolve_pending_comment(state, word, node["comment_url"])
            save_progress(state)
            print(f"[FILTERED] Checked {checked}; comment did not meet the Persian/word/bot filters.", flush=True)
            continue
        if row["comment_id"] in ids or normalize_text(row["text"]) in texts:
            resolve_pending_comment(state, word, node["comment_url"])
            save_progress(state)
            print(f"[DUPLICATE] Checked {checked}; comment already saved for this word.", flush=True)
            continue
        state["comments"].append(row)
        resolve_pending_comment(state, word, node["comment_url"])
        ids.add(row["comment_id"])
        texts.add(normalize_text(row["text"]))
        complete = len(ids) >= COMMENTS_PER_WORD
        record_status(state, word, "complete" if complete else "partial")
        save_progress(state)
        print(f"[SAVED] {len(ids)}/{COMMENTS_PER_WORD}: {row['comment_url']} "
              f"| CSV: {OUTPUT_CSV}", flush=True)
        if complete:
            break
    count = len(rows_for_word(state, word))
    status = "complete" if count >= COMMENTS_PER_WORD else "partial" if count else "no_match"
    record_status(state, word, status, "" if status == "complete" else
                  "No more qualifying comments within configured search limits; retry or increase limits.")
    print(f"[WORD DONE] Checked {checked}; {count}/{COMMENTS_PER_WORD} saved; {status}.", flush=True)


def collect_comments(driver, words, state):
    """Return (state, driver); driver is a new browser if the old one stopped responding."""
    consecutive_errors = 0
    try:
        for index, word in enumerate(words, 1):
            existing = rows_for_word(state, word)
            if len(existing) >= COMMENTS_PER_WORD:
                record_status(state, word, "complete")
                continue
            print(f"[{index}/{len(words)}] {word}: {len(existing)}/{COMMENTS_PER_WORD} saved", flush=True)
            try:
                for restart in range(MAX_BROWSER_RESTARTS + 1):
                    try:
                        collect_word(driver, word, state)
                        break
                    except BrowserUnresponsive as error:
                        if restart == MAX_BROWSER_RESTARTS:
                            raise
                        print(f"[RESTART] {str(error)[:200]} Restarting the browser and retrying this word.",
                              flush=True)
                        driver = restart_browser(driver)
                        print(f"[RESTART] Waiting {RESTART_WAIT}s before retrying.", flush=True)
                        time.sleep(RESTART_WAIT)
                consecutive_errors = 0
            except RedditAccessBlocked as error:
                record_status(state, word, "blocked", str(error))
                print("Stopped:", error)
                break
            except WebDriverException as error:
                record_status(state, word, "error", str(error)[:1000])
                print("Browser/page error:", str(error)[:400])
                consecutive_errors += 1
                if (isinstance(error, RedditPageLoadError) or not isinstance(error, TimeoutException)
                        or consecutive_errors >= MAX_CONSECUTIVE_ERRORS):
                    print("Stopping; inspect the browser and restart it if needed.")
                    break
            except KeyboardInterrupt:
                record_status(state, word, "interrupted", "Interrupted by user; accepted comments are saved.")
                print("Interrupted; progress saved.")
                break
            finally:
                save_progress(state)
            if index < len(words):
                time.sleep(DELAY_BETWEEN_WORDS)
    except KeyboardInterrupt:
        print("Interrupted between words; progress saved.")
    finally:
        save_progress(state)
    return state, driver


In [ ]:
state = load_progress()
save_progress(state)
print(f"Checkpoint contains {len(state['comments'])} comments.")
# Close only a previous browser created by this notebook when this cell is rerun.
if globals().get("driver") is not None:
    try:
        driver.quit()
    except WebDriverException as error:
        print("Previous browser was already closed:", error.msg[:150])
driver = start_browser()

In [ ]:
# Rerun this cell to resume from the checkpoint.
state = load_progress()
words_to_collect = unsearched_words(state, selected_words)
print(f"{len(words_to_collect)} words not searched yet; "
      f"{len(selected_words) - len(words_to_collect)} already searched (incomplete ones: use the Retry cell).")
state, driver = collect_comments(driver, words_to_collect, state)
all_comments = state["comments"]
reddit_df = pd.DataFrame(all_comments, columns=CSV_COLUMNS)
status_df = pd.DataFrame(list(state["words"].values()), columns=STATUS_COLUMNS)
print(f"Saved {len(reddit_df)} comments to {OUTPUT_CSV.resolve()}")
print(status_df["status"].value_counts().to_string() if not status_df.empty else "No words processed.")
pending_count = sum(len(items) for items in state.get("pending_comments", {}).values())
hung_count = sum(entry.get("browser_hang", False)
                 for items in state.get("pending_comments", {}).values() for entry in items.values())
print(f"Unreadable links retained for retry: {pending_count - hung_count}")
print(f"Links skipped because they froze the browser: {hung_count}")
reddit_df.head(20)

## 7. Retry incomplete words

Set `RETRY_INCOMPLETE = True` and run this cell after the main collection has gone through all words. It retries words with fewer than three comments and reopens links that previously froze the browser.

In [ ]:
RETRY_INCOMPLETE = False
if RETRY_INCOMPLETE:
    state = load_progress()
    for items in state.get("pending_comments", {}).values():
        for entry in items.values():
            entry.pop("browser_hang", None)  # give links that froze the browser one more chance
    incomplete_words = [word for word in selected_words
                        if len(rows_for_word(state, word)) < COMMENTS_PER_WORD]
    state, driver = collect_comments(driver, incomplete_words, state)
    reddit_df = pd.DataFrame(state["comments"], columns=CSV_COLUMNS)
    status_df = pd.DataFrame(list(state["words"].values()), columns=STATUS_COLUMNS)
else:
    print("Optional retry skipped.")
status_df

## 8. Close the browser

Set `CLOSE_BROWSER = True` and run this cell when done. The checkpoint and browser profile are kept for the next run.

In [ ]:
CLOSE_BROWSER = False
if CLOSE_BROWSER and globals().get("driver") is not None:
    try:
        driver.quit()
    finally:
        driver = None